In [3]:
import os
import sys
sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints.chat_models import ChatNVIDIA

from rag_core import (
    Embedding,
    VectorStore,
    RAGRetreiver,
    QueryRetreiver,
    CaptureFlow,
    load_docs_with_category,
    invoke_with_retry,
)
 
load_dotenv()
nvidia_api_key = os.getenv("NVIDIA_API_KEY")

In [4]:
documents = load_docs_with_category("../docs")
print(f"Loaded {len(documents)} document")

Loaded 809 documents across categories
Loaded 809 document


In [5]:
embedding = Embedding()
vector_store = VectorStore() 
vector_store.reset_collection()
texts = [d.page_content for d in documents]
embeddings = embedding.generate_embeddings(texts)
vector_store.add_docs(documents, embeddings)
 
rag_retreiver = RAGRetreiver(vector_store, embedding)
 
print("Knowledge base size:", vector_store.collection.count())

Embeddings Model Name: nvidia/nemotron-3-embed-1b
Model loaded successfully.


c:\Users\win11\OneDrive\Desktop\AAI_learning_with_development\venv\Lib\site-packages\langchain_nvidia_ai_endpoints\_common.py:250: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


Collection reset.
Embedded batch 1 (10 docs)
Embedded batch 2 (10 docs)
Embedded batch 3 (10 docs)
Embedded batch 4 (10 docs)
Embedded batch 5 (10 docs)
Embedded batch 6 (10 docs)
Embedded batch 7 (10 docs)
Embedded batch 8 (10 docs)
Embedded batch 9 (10 docs)
Embedded batch 10 (10 docs)
Embedded batch 11 (10 docs)
Embedded batch 12 (10 docs)
Embedded batch 13 (10 docs)
Embedded batch 14 (10 docs)
Embedded batch 15 (10 docs)
Embedded batch 16 (10 docs)
Embedded batch 17 (10 docs)
Embedded batch 18 (10 docs)
Embedded batch 19 (10 docs)
Embedded batch 20 (10 docs)
Embedded batch 21 (10 docs)
Embedded batch 22 (10 docs)
Embedded batch 23 (10 docs)
Embedded batch 24 (10 docs)
Embedded batch 25 (10 docs)
Embedded batch 26 (10 docs)
Embedded batch 27 (10 docs)
Embedded batch 28 (10 docs)
Embedded batch 29 (10 docs)
Embedded batch 30 (10 docs)
Embedded batch 31 (10 docs)
Embedded batch 32 (10 docs)
Embedded batch 33 (10 docs)
Embedded batch 34 (10 docs)
Embedded batch 35 (10 docs)
Embedded ba

In [ ]:
llm = ChatNVIDIA(
    model="mistralai/mistral-nemotron",
    nvidia_api_key=nvidia_api_key
)

c:\Users\win11\OneDrive\Desktop\AAI_learning_with_development\venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: UserWarning: WARNING! max_retries is not default parameter.
                max_retries was transferred to model_kwargs.
                Please confirm that max_retries is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
query_retriever = QueryRetreiver(rag_retreiver)
 
answer_junior = query_retriever._rag_advanced(
    "Rolling mill high vibration what should I check first",
    role="junior", llm=llm
)
print("JUNIOR:\n", answer_junior)
 
answer_senior = query_retriever._rag_advanced(
    "Rolling mill high vibration what should I check first",
    role="senior", llm=llm
)
print("\nSENIOR:\n", answer_senior)

Embedded batch 1 (1 docs)
LLM call failed (attempt 1/5): HTTPSConnectionPool(host='integrate.api.nvidia.com', port=443): Read timed out. (read timeout=120.0)
Retrying in 5s...
JUNIOR:
 1. **Check the strip temperature uniformity** – If the vibration only happens with thin gauges, measure the temperature across the strip width before and after the roll stand to ensure it is even. Uneven cooling can cause thermal distortion in the rolls, leading to chatter. [Source: log-hsm-chatter-vib.txt]

2. **Look for material buildup** – If vibration changes with speed and leaves repeating marks on the strip, inspect the rolls for dross, scale, or debris buildup. Clean them **only during planned shutdowns** using approved methods—never scrape while the mill is running. [Source: galvanizing-line-sink-roll-vibration.txt]

3. **Confirm process conditions** – Before assuming a mechanical issue, check if suction lines are restricted, if fluid levels are low, or if lubrication is inadequate. A high vibrat

In [8]:
capture_flow = CaptureFlow(vector_store, embedding, llm)
 
confirmation = capture_flow.capture(
    "pressure valve issue on Furnace 2, caused by a stuck relief valve after "
    "a cold start, solved by manually cycling the valve twice before ignition"
)
print(confirmation)

Embedded batch 1 (1 docs)
Added 1 documents to the vector store.
Got it — logged as:
Title: Stuck Relief Valve in Furnace 2 After Cold Start
Category: furnace
Solution: Manually cycled the relief valve twice before ignition to resolve the issue.


In [9]:
answer = query_retriever._rag_advanced(
    "Furnace 2 pressure valve issue after cold start", role="senior", llm=llm
)
print(answer)

Embedded batch 1 (1 docs)
The issue with the pressure valve in Furnace 2 after a cold start was caused by a **stuck relief valve** [Source: captured_Stuck_Relief_Valve_in_Furnace_.txt]. The solution involved manually cycling the relief valve twice before ignition to resolve the issue.

### Key Points:
1. **Symptom**: Pressure valve malfunction due to a stuck relief valve post-cold startup.
2. **Root Cause**: Likely related to thermal expansion or debris challenging proper closure/open operation under cold conditions.
3. **Mitigation**: Manual cycling forced seating and ensured valve mobility before ignition.

### Non-Obvious Considerations:
- **Cold Start Dynamics**: Stiffness or reduced lubrication effects due to low temperature may exacerbate sticking. Inspect seals and pivot mechanisms for wear or fouling. [Source: captured_Stuck_Relief_Valve_in_Furnace_.txt]
- **Pressure Tap Validation**: If instrument readings appear stable but flames are present (as in [Source: rare-failure-furna

In [10]:
edge_case_queries = [
    "",                                          # empty query
    "asdkjaslkdj random gibberish query",        # nonsense
    "What is the capital of France?",            # off-topic
    "vibration",                                 # single word, vague
    "Rolling mill station 3 showing high vibration — what should I check first?" * 5,  # long
]
 
for q in edge_case_queries:
    print(f"\n--- Query: {q[:60]!r} ---")
    try:
        ans = query_retriever._rag_advanced(q, role="junior", llm=llm)
        print(ans[:300])
    except Exception as e:
        print(f"ERROR: {e}")


--- Query: '' ---
I couldn't find relevant information in the knowledge base to answer that.

--- Query: 'asdkjaslkdj random gibberish query' ---
Embedded batch 1 (1 docs)
I couldn't find relevant information in the knowledge base to answer that.

--- Query: 'What is the capital of France?' ---
Embedded batch 1 (1 docs)
I couldn't find relevant information in the knowledge base to answer that.

--- Query: 'vibration' ---
Embedded batch 1 (1 docs)
**Step-by-Step Guide to Checking Vibration Issues**

### **1. Safety First**
- **Stop safe operations if needed**: follow shut-down procedures, use isolation or exclusion if working in hot or hazardous areas. Do not touch hot surfaces or moving parts.
- **Never use improvised tools on hot rolls or m

--- Query: 'Rolling mill station 3 showing high vibration — what should ' ---
Embedded batch 1 (1 docs)
**Step 1: Check Process Conditions First** [Source: technician-reference-reading-vibration-trends.txt]
- Before inspecting the machine, confir